Packages importeren en zorgen dat output niet wordt onderbroken door puntjes

In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import altair as alt
import pickle
import gdown
import matplotlib.pyplot as plt

Pickle file ophalen

In [ ]:
with open('../data/supermarket_combined_sales.pkl', 'rb') as pickle_file:
    sales = pickle.load(pickle_file)

combined_sales  = sales['combined_sales']

Overige data ophalen

In [ ]:
shared_url = f'https://drive.google.com/file/d/10Q0ZCqkqt91xQU4VRvFHdatxNkUi8A2E/view?usp=drive_link'

file_id = shared_url.split('/d/')[1].split('/')[0]
download_url = f'https://drive.google.com/uc?id={file_id}'

output = 'items.parquet'
gdown.download(download_url, output, quiet=False)

items = pd.read_parquet(output)
print(items.head())

shared_url = f'https://drive.google.com/file/d/1yLFRvstJ2-XbeTIGfe6i47l4z_qu8iY8/view?usp=drive_link'

file_id = shared_url.split('/d/')[1].split('/')[0]
download_url = f'https://drive.google.com/uc?id={file_id}'

output = 'stores.parquet'
gdown.download(download_url, output, quiet=False)

stores = pd.read_parquet(output)
print(stores.head())

Verkleinen geheugengebruik door types aan te passen

In [ ]:
for col in combined_sales.select_dtypes(include=["object"]).columns:
    combined_sales[col] = combined_sales[col].astype("category")

for col in combined_sales.select_dtypes(include=["number"]).columns:
    combined_sales[col] = pd.to_numeric(combined_sales[col], downcast="integer")
    combined_sales[col] = pd.to_numeric(combined_sales[col], downcast="float")

for col in items.select_dtypes(include=["object"]).columns:
    items[col] = items[col].astype("category")

for col in items.select_dtypes(include=["number"]).columns:
    items[col] = pd.to_numeric(items[col], downcast="integer")
    items[col] = pd.to_numeric(items[col], downcast="float")

for col in stores.select_dtypes(include=["object"]).columns:
    stores[col] = stores[col].astype("category")

for col in stores.select_dtypes(include=["number"]).columns:
    stores[col] = pd.to_numeric(stores[col], downcast="integer")
    stores[col] = pd.to_numeric(stores[col], downcast="float")

Combineren van sales met items en stores

In [ ]:
combined_sales_items = pd.merge(combined_sales, items, on='item_nbr', how='left')
print(combined_sales_items.shape)
print()
print()


In [ ]:
combined_sales_items_stores = pd.merge(combined_sales_items, stores, on='store_nbr', how='left')

In [ ]:
combined_sales.to_parquet("combined_history_items_stores.parquet", index=False)

Kerngetallen printen

In [ ]:
print(combined_sales_items_stores.head())
print()

print(combined_sales_items_stores.shape)
print()

print(combined_sales_items_stores['onpromotion'].value_counts())
print()

missende_aantal = combined_sales_items_stores.isnull().sum()
missende_aantal = missende_aantal[missende_aantal > 0]
print(missende_aantal)


Grafiek maken van verkopen per maand

In [ ]:
monthly_sales = combined_sales_items_stores.groupby('month')['unit_sales'].sum()



In [ ]:
monthly_sales.plot(kind='line', marker='o', title='Verkoop per Maand')
plt.xlabel('Maand')
plt.ylabel('Verkoop')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Opslaan pickle file

In [ ]:
dc_sales_history = {
'combined_sales_items_stores': combined_sales_items_stores,
'items': items,
'stores': stores,
}

# Save dc_exercise_1_2_3 as 'dc-ames-housing-pieter-exercise-1-2-3.pkl'
with open('../data/supermarket_combined_sales.pkl', 'wb') as pickle_file:
    pickle.dump(dc_sales_history, pickle_file)